# AIC365 — Natural Language Processing
## LAB 4 TASKS: Text Normalization, Cleaning & Minimum Edit Distance

**Student:** MUSA SAJJAD  
**Registration No.:** SP24-BDS-039  


In [1]:
# Install/download required NLP resources
!pip -q install spacy nltk
!python -m spacy download en_core_web_sm -q

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('words', quiet=True)

print("Setup complete.")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Setup complete.


In [2]:
# Imports
import re
import string
import pandas as pd
import nltk
import spacy

from nltk.corpus import stopwords, words as nltk_words
from nltk.stem import PorterStemmer
from IPython.display import display

nlp = spacy.load("en_core_web_sm")
nltk_stop_words = set(stopwords.words("english"))
english_words = set(w.lower() for w in nltk_words.words())
stemmer = PorterStemmer()

pd.set_option("display.max_colwidth", None)

## Dataset — `support_chats`

In [3]:
# Self-contained support chat dataset.
# It deliberately includes the example misspellings listed in the Lab 4 task.

support_chats = pd.DataFrame({
    "message_id": range(1, 11),
    "text": [
        "pls chek my pakage, it has not arived yet!",
        "I beleive the payment was deducted twice from my account.",
        "The app is definately slower after the latest update.",
        "u sent the wrong item, pls arrange delievery of the correct one asap.",
        "My application keeps crashin whenever I open the orders page.",
        "can u chek wich plan includes international shipping?",
        "I recieved a damaged product and need a replacement.",
        "The issue occured again after I restarted the phone.",
        "pls send me a seperate msg when the refund is processed.",
        "I cannot beleive the pakage is still not here."
    ]
})

display(support_chats)

,message_id,text
0,1,"pls chek my pakage, it has not arived yet!"
1,2,I beleive the payment was deducted twice from my account.
2,3,The app is definately slower after the latest update.
3,4,"u sent the wrong item, pls arrange delievery of the correct one asap."
4,5,My application keeps crashin whenever I open the orders page.
5,6,can u chek wich plan includes international shipping?
6,7,I recieved a damaged product and need a replacement.
7,8,The issue occured again after I restarted the phone.
8,9,pls send me a seperate msg when the refund is processed.
9,10,I cannot beleive the pakage is still not here.


## Activity 1 — Clean `support_chats` end to end

Required by the manual:
- lowercase
- remove stop words
- lemmatize
- report vocabulary size before and after cleaning
- calculate percentage reduction

In [4]:
def basic_normalize(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def full_text_pipeline(text):
    text = basic_normalize(text)
    doc = nlp(text)
    cleaned_tokens = [
        token.lemma_.lower()
        for token in doc
        if token.text.lower() not in nltk_stop_words
        and not token.is_space
    ]
    return " ".join(cleaned_tokens)

support_chats["cleaned_text"] = support_chats["text"].apply(full_text_pipeline)

raw_vocab = set(basic_normalize(" ".join(support_chats["text"])).split())
clean_vocab = set(" ".join(support_chats["cleaned_text"]).split())

vocab_before = len(raw_vocab)
vocab_after = len(clean_vocab)
reduction = ((vocab_before - vocab_after) / vocab_before) * 100

print("Vocabulary size before cleaning:", vocab_before)
print("Vocabulary size after cleaning :", vocab_after)
print(f"Percentage reduction           : {reduction:.2f}%")
display(support_chats[["message_id", "text", "cleaned_text"]])

Vocabulary size before cleaning: 70
Vocabulary size after cleaning : 50
Percentage reduction           : 28.57%


,message_id,text,cleaned_text
0,1,"pls chek my pakage, it has not arived yet!",pls chek pakage arive yet
1,2,I beleive the payment was deducted twice from my account.,beleive payment deduct twice account
2,3,The app is definately slower after the latest update.,app definately slow late update
3,4,"u sent the wrong item, pls arrange delievery of the correct one asap.",u send wrong item pls arrange delievery correct one asap
4,5,My application keeps crashin whenever I open the orders page.,application keep crashin whenever open order page
5,6,can u chek wich plan includes international shipping?,u chek wich plan include international shipping
6,7,I recieved a damaged product and need a replacement.,recieve damage product need replacement
7,8,The issue occured again after I restarted the phone.,issue occur restart phone
8,9,pls send me a seperate msg when the refund is processed.,pls send seperate msg refund process
9,10,I cannot beleive the pakage is still not here.,beleive pakage still


## Activity 2 — Porter Stemmer vs spaCy Lemmatization

Pick 10 words from the cleaned support-chat vocabulary, apply both methods, and flag stems that are not real English words.

In [5]:
# Prefer useful words if they exist; otherwise fill from the cleaned vocabulary.
preferred = [
    "shipping", "processed", "payment", "damaged", "replacement",
    "international", "application", "orders", "latest", "slower"
]
selected_words = [w for w in preferred if w in clean_vocab]

for w in sorted(clean_vocab):
    if w.isalpha() and w not in selected_words and len(selected_words) < 10:
        selected_words.append(w)

selected_words = selected_words[:10]

rows = []
for word in selected_words:
    stem = stemmer.stem(word)
    doc = nlp(word)
    lemma = doc[0].lemma_
    stem_is_real = stem.lower() in english_words

    rows.append({
        "Original Word": word,
        "Porter Stem": stem,
        "spaCy Lemma": lemma,
        "Stem is real English word?": "Yes" if stem_is_real else "No"
    })

comparison_df = pd.DataFrame(rows)
display(comparison_df)

print("\nFlagged non-dictionary stems:")
display(comparison_df[comparison_df["Stem is real English word?"] == "No"])

,Original Word,Porter Stem,spaCy Lemma,Stem is real English word?
0,shipping,ship,shipping,Yes
1,payment,payment,payment,Yes
2,replacement,replac,replacement,No
3,international,intern,international,Yes
4,application,applic,application,No
5,account,account,account,Yes
6,app,app,app,No
7,arive,ariv,arive,No
8,arrange,arrang,arrange,No
9,asap,asap,asap,No



Flagged non-dictionary stems:


,Original Word,Porter Stem,spaCy Lemma,Stem is real English word?
2,replacement,replac,replacement,No
4,application,applic,application,No
6,app,app,app,No
7,arive,ariv,arive,No
8,arrange,arrang,arrange,No
9,asap,asap,asap,No


## Activity 3 — Spelling suggestion using `nltk.edit_distance`

The manual gives example misspellings such as:
`pakage, arived, chek, beleive, definately, crashin, delievery, wich, recieved, occured, seperate`.

For each misspelling, the function finds the vocabulary word with the smallest Levenshtein edit distance and then produces corrected versions of all chat messages.

In [6]:
# Vocabulary supplied/extended from the lab's spelling-correction idea.
# Extra words are included so every example typo in the support chats has a sensible target.

vocab_list = [
    "package", "arrived", "check", "believe", "definitely", "crashing",
    "delivery", "which", "received", "occurred", "separate",
    "receive", "until", "because", "achieve", "necessary",
    "please", "you", "message", "asap"
]

misspelled_words = [
    "pakage", "arived", "chek", "beleive", "definately",
    "crashin", "delievery", "wich", "recieved", "occured", "seperate"
]

def spelling_suggestion(word, vocabulary):
    best_match = None
    min_distance = float("inf")

    for candidate in vocabulary:
        distance = nltk.edit_distance(word.lower(), candidate.lower())
        if distance < min_distance:
            min_distance = distance
            best_match = candidate

    return best_match, min_distance

correction_map = {}
correction_rows = []

for wrong in misspelled_words:
    suggestion, distance = spelling_suggestion(wrong, vocab_list)
    correction_map[wrong] = suggestion
    correction_rows.append({
        "Misspelled Word": wrong,
        "Suggested Correction": suggestion,
        "Edit Distance": distance
    })

corrections_df = pd.DataFrame(correction_rows)
display(corrections_df)

,Misspelled Word,Suggested Correction,Edit Distance
0,pakage,package,1
1,arived,arrived,1
2,chek,check,1
3,beleive,believe,2
4,definately,definitely,1
5,crashin,crashing,1
6,delievery,delivery,1
7,wich,which,1
8,recieved,received,2
9,occured,occurred,1


In [7]:
def replace_words(text, mapping):
    # Replace whole words only, preserving surrounding punctuation.
    pattern = re.compile(r"\b(" + "|".join(map(re.escape, mapping.keys())) + r")\b", flags=re.IGNORECASE)

    def repl(match):
        original = match.group(0)
        replacement = mapping[original.lower()]
        if original.istitle():
            return replacement.title()
        if original.isupper():
            return replacement.upper()
        return replacement

    return pattern.sub(repl, text)

support_chats["corrected_spelling"] = support_chats["text"].apply(
    lambda x: replace_words(x, correction_map)
)

print("Corrected version of every chat message:")
display(support_chats[["message_id", "text", "corrected_spelling"]])

Corrected version of every chat message:


,message_id,text,corrected_spelling
0,1,"pls chek my pakage, it has not arived yet!","pls check my package, it has not arrived yet!"
1,2,I beleive the payment was deducted twice from my account.,I believe the payment was deducted twice from my account.
2,3,The app is definately slower after the latest update.,The app is definitely slower after the latest update.
3,4,"u sent the wrong item, pls arrange delievery of the correct one asap.","u sent the wrong item, pls arrange delivery of the correct one asap."
4,5,My application keeps crashin whenever I open the orders page.,My application keeps crashing whenever I open the orders page.
5,6,can u chek wich plan includes international shipping?,can u check which plan includes international shipping?
6,7,I recieved a damaged product and need a replacement.,I received a damaged product and need a replacement.
7,8,The issue occured again after I restarted the phone.,The issue occurred again after I restarted the phone.
8,9,pls send me a seperate msg when the refund is processed.,pls send me a separate msg when the refund is processed.
9,10,I cannot beleive the pakage is still not here.,I cannot believe the package is still not here.


## Activity 4 — Live-chat autocorrect simulation

For **messages 1, 4, 6, and 9**, correct misspelled **and informal** words and show the final version a support agent would see.

In [8]:
informal_map = {
    "pls": "please",
    "u": "you",
    "msg": "message",
    "asap": "as soon as possible"
}

autocorrect_map = {**correction_map, **informal_map}

selected_ids = [1, 4, 6, 9]
autocorrect_rows = []

for message_id in selected_ids:
    original = support_chats.loc[
        support_chats["message_id"] == message_id, "text"
    ].iloc[0]

    corrected = replace_words(original, autocorrect_map)
    corrected = corrected[0].upper() + corrected[1:] if corrected else corrected

    autocorrect_rows.append({
        "Message ID": message_id,
        "Original Message": original,
        "Autocorrected Message": corrected
    })

autocorrect_df = pd.DataFrame(autocorrect_rows)
display(autocorrect_df)

,Message ID,Original Message,Autocorrected Message
0,1,"pls chek my pakage, it has not arived yet!","Please check my package, it has not arrived yet!"
1,4,"u sent the wrong item, pls arrange delievery of the correct one asap.","You sent the wrong item, please arrange delivery of the correct one as soon as possible."
2,6,can u chek wich plan includes international shipping?,Can you check which plan includes international shipping?
3,9,pls send me a seperate msg when the refund is processed.,Please send me a separate message when the refund is processed.
